## 1. Imports

In [13]:
from __future__ import annotations
import random
import copy                             
import time
import numpy as np
from pathlib import Path

from deap import base, creator, tools

# === DRY IMPORTS FROM schedule_engine/notebooks/ ===
from schedule_engine.notebooks.core import load_data, create_random_individual
from schedule_engine.notebooks.core import course_aware_crossover, smart_mutation
from schedule_engine.notebooks.core import create_evaluator, get_constraint_breakdown
from schedule_engine.notebooks.core import EvolutionConfig, setup_deap, get_best_individual, EvolutionStats
from schedule_engine.notebooks.viz import plot_convergence, plot_constraint_breakdown, print_summary
from schedule_engine.notebooks.strategies import local_search_individual

print(" All imports successful!")

 All imports successful!


## 2. Mode B Configuration

In [14]:

# MODE B CONFIGURATION

from datetime import datetime

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# GA Parameters
POP_SIZE = 10
NGEN = 100
CXPB = 0.9
MUTPB = 0.2
FITNESS_WEIGHTS = (-1.0, -0.01)

# MODE B SPECIFIC: Local search parameters
LOCAL_SEARCH_PROB = 0.2        # Probability of LS per individual per gen
LOCAL_SEARCH_ITERATIONS = 10   # Iterations when LS is applied

# Paths - Organized by mode with timestamp
TIMESTAMP = datetime.now().strftime("%Y%m%d_%H%M%S")
DATA_DIR = Path("../data")
OUTPUT_DIR = Path(f"../output/mode_b_memetic/{TIMESTAMP}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f" Mode B Config: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
print(f" Output: {OUTPUT_DIR}")

 Mode B Config: pop=10, ngen=100, LS_prob=0.2
 Output: ../output/mode_b_memetic/20260122_204458


## 3. Load Data

In [15]:
data = load_data(
    data_dir=DATA_DIR,
    opening_time="10:00",
    closing_time="17:00",
    closed_days=["Saturday"],
)

print(f" {data.summary()}")

[!warn] groups enrolled but courses missing

CE604: BCE5A, BCE5B, BCE5C, BCE5D, BCE5E, BCE5F (ltp null)

ENCE 256: BCE4A, BCE4B, BCE4C, BCE4D, BCE4E, BCE4F (ltp null)

ENIE 254: BIE4A, BIE4B (ltp null)

ME706: BME7A, BME7B (ltp null)

16 course enrollments skipped

 Courses: 668, Groups: 74, Instructors: 181, Rooms: 67, Quanta: 42


## 4. Test Components

In [16]:
# Test individual creation and evaluation
evaluate = create_evaluator(data)
test_ind = create_random_individual(data)
print(f" Individual: {len(test_ind)} genes")
print(f" Initial fitness: hard={evaluate(test_ind)[0]}")

# Test local search
improved_ind, improvement = local_search_individual(
    test_ind, data, evaluate, max_iterations=5
)
print(f" After LS: hard={evaluate(improved_ind)[0]} (improvement={improvement})")

 Individual: 713 genes
 Initial fitness: hard=5364.0
 After LS: hard=5359.0 (improvement=5.0)


## 5. Memetic NSGA-II Evolution (Mode B Specific)

In [17]:
def run_memetic_nsga2():
    """Run NSGA-II with memetic local search."""
    print(f" Memetic NSGA-II: pop={POP_SIZE}, ngen={NGEN}, LS_prob={LOCAL_SEARCH_PROB}")
    start = time.time()
    
    # Setup DEAP
    setup_deap(FITNESS_WEIGHTS)
    
    toolbox = base.Toolbox()
    toolbox.register("individual", lambda: creator.Individual(create_random_individual(data)))
    toolbox.register("population", tools.initRepeat, list, toolbox.individual)
    toolbox.register("evaluate", evaluate)
    toolbox.register("mate", course_aware_crossover)
    toolbox.register("mutate", lambda ind: smart_mutation(ind, data))
    toolbox.register("select", tools.selNSGA2)
    
    # Initialize population
    pop = toolbox.population(n=POP_SIZE)
    for ind in pop:
        ind.fitness.values = toolbox.evaluate(ind)
    
    stats = EvolutionStats()
    
    for gen in range(NGEN):
        # Standard NSGA-II selection + variation
        offspring = [copy.deepcopy(ind) for ind in toolbox.select(pop, len(pop))]
        
        # Crossover
        for i in range(0, len(offspring)-1, 2):
            if random.random() < CXPB:
                toolbox.mate(offspring[i], offspring[i+1])
                del offspring[i].fitness.values
                del offspring[i+1].fitness.values
        
        # Mutation
        for ind in offspring:
            if random.random() < MUTPB:
                toolbox.mutate(ind)
                del ind.fitness.values
        
        # === MODE B SPECIFIC: Local Search ===
        for ind in offspring:
            if random.random() < LOCAL_SEARCH_PROB:
                genes = list(ind)
                improved_genes, _ = local_search_individual(
                    genes, data, evaluate, LOCAL_SEARCH_ITERATIONS
                )
                ind[:] = improved_genes
                del ind.fitness.values
        
        # Evaluate
        for ind in offspring:
            if not ind.fitness.valid:
                ind.fitness.values = toolbox.evaluate(ind)
        
        # Survivor selection
        pop = toolbox.select(pop + offspring, POP_SIZE)
        
        # Stats
        hard_vals = [ind.fitness.values[0] for ind in pop]
        soft_vals = [ind.fitness.values[1] for ind in pop]
        stats.generations.append(gen)
        stats.min_hard.append(float(min(hard_vals)))
        stats.avg_hard.append(float(np.mean(hard_vals)))
        stats.max_hard.append(float(max(hard_vals)))
        stats.feasible_count.append(sum(1 for h in hard_vals if h == 0))
        stats.min_soft.append(float(min(soft_vals)))
        stats.avg_soft.append(float(np.mean(soft_vals)))
        
        # Show progress every generation
        print(f"  Gen {gen:3d}: min_hard={stats.min_hard[-1]:3.0f}, min_soft={stats.min_soft[-1]:5.0f}, feasible={stats.feasible_count[-1]}/{POP_SIZE}")
    
    stats.elapsed_time = time.time() - start
    print(f" Done in {stats.elapsed_time:.1f}s")
    return pop, stats

# RUN
final_pop, stats = run_memetic_nsga2()

 Memetic NSGA-II: pop=10, ngen=100, LS_prob=0.2
  Gen   0: min_hard=4536, min_soft= 2533, feasible=0/10
  Gen   1: min_hard=4112, min_soft= 2501, feasible=0/10
  Gen   2: min_hard=3900, min_soft= 2308, feasible=0/10
  Gen   3: min_hard=3900, min_soft= 2308, feasible=0/10
  Gen   4: min_hard=3616, min_soft= 2183, feasible=0/10
  Gen   5: min_hard=3616, min_soft= 2163, feasible=0/10
  Gen   6: min_hard=3516, min_soft= 1847, feasible=0/10
  Gen   7: min_hard=3166, min_soft= 1843, feasible=0/10
  Gen   8: min_hard=3079, min_soft= 1650, feasible=0/10
  Gen   9: min_hard=3079, min_soft= 1650, feasible=0/10
  Gen  10: min_hard=2827, min_soft= 1575, feasible=0/10
  Gen  11: min_hard=2595, min_soft= 1575, feasible=0/10
  Gen  12: min_hard=2502, min_soft= 1514, feasible=0/10
  Gen  13: min_hard=2422, min_soft= 1479, feasible=0/10
  Gen  14: min_hard=2287, min_soft= 1370, feasible=0/10
  Gen  15: min_hard=2287, min_soft= 1336, feasible=0/10
  Gen  16: min_hard=2278, min_soft= 1258, feasible=0/10


## 6. Results & Visualization

In [18]:
best = get_best_individual(final_pop)
breakdown = get_constraint_breakdown(best, data)

print_summary(final_pop, stats, breakdown)

plot_convergence(stats, OUTPUT_DIR / "mode_b_convergence.png", title_prefix="Mode B: ")
plot_constraint_breakdown(breakdown, OUTPUT_DIR / "mode_b_breakdown.png", title="Mode B: Constraint Violations")


 EVOLUTION SUMMARY

 Best Solution:
   Hard Violations: 1569
   Soft Penalty:    736.0
   Feasible:         No

 Final Population (n=10):
   Feasible:     0/10 (0.0%)
   Min Hard:     1569
   Avg Hard:     1599.5
   Min Soft:     709.0
   Avg Soft:     731.5

️ Execution Time: 29.6s

 Best Solution Constraint Breakdown:
    course_completeness: 0
    instructor_exclusivity: 126
    instructor_qualifications: 17
    instructor_schedule_compactness: 99
    instructor_time_availability: 494
    paired_cohort_practical_alignment: 0
    room_exclusivity: 274
    room_suitability: 1
    room_time_availability: 0
    session_continuity: 65
    student_group_exclusivity: 657
    student_lunch_break: 306
    student_schedule_compactness: 266

   Total Hard: 1075, Total Soft: 1230.0

 Saved: ../output/mode_b_memetic/20260122_204458/mode_b_convergence.png
 Saved: ../output/mode_b_memetic/20260122_204458/mode_b_breakdown.png


/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:93: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
/home/krishna/Desktop/schedule-engine/src/schedule_engine/notebooks/viz.py:173: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
